# HierarchicalDet — Kaggle Training Notebook

Trains all three tiers of HierarchicalDet on the DENTEX dataset.

**Session plan (Kaggle T4 = 12h limit):**
```
Session 1: TIER_TO_TRAIN=1, PREV_OUTPUT_DIR=None
Session 2: TIER_TO_TRAIN=2, PREV_OUTPUT_DIR='/kaggle/input/<s1-output>/output'
Session 3: TIER_TO_TRAIN=3, PREV_OUTPUT_DIR='/kaggle/input/<s2-output>/output'
```
After each session: **Save Version → Save & Run All** to commit `/kaggle/working/` as output.

In [ ]:
# ── CONFIGURATION — edit before running ───────────────────────────────────────

GITHUB_REPO     = 'https://github.com/AIscend-Research/HierarchicalDet'
TIER_TO_TRAIN   = 1      # 1, 2, or 3
MAX_ITER        = 40000
NUM_GPUS        = 1

# Previous session output (None for fresh tier-1 start)
# e.g. '/kaggle/input/hierarchicaldet-tier1/output'
PREV_OUTPUT_DIR = None

# Pre-uploaded DENTEX zips as a Kaggle dataset (saves ~30 min per session)
# e.g. '/kaggle/input/dentex-dataset'
KAGGLE_DENTEX_INPUT = None

# HuggingFace token (optional — only needed if KAGGLE_DENTEX_INPUT is None)
HF_TOKEN = None

# ── Derived paths (do not edit) ───────────────────────────────────────────────
import os
WORK_DIR     = '/kaggle/working'
REPO_DIR     = f'{WORK_DIR}/HierarchicalDet'
OFFICIAL_DIR = f'{WORK_DIR}/HierarchicalDet_official'
DATA_ROOT    = f'{WORK_DIR}/sorted/challenge'
OUTPUT_DIR   = f'{WORK_DIR}/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Tier:            {TIER_TO_TRAIN}')
print(f'Previous output: {PREV_OUTPUT_DIR}')
print(f'DENTEX source:   {KAGGLE_DENTEX_INPUT or "HuggingFace download"}')

In [ ]:
# ── 0. Hardware check ─────────────────────────────────────────────────────────
import torch

print('=== HARDWARE ===')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!free -h | grep Mem
!df -h /kaggle/working | tail -1
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.version.cuda}')
assert torch.cuda.is_available(), 'No GPU! Set Accelerator → GPU T4 in Settings.'
print(f'GPU:     {torch.cuda.get_device_name(0)}')

In [ ]:
# ── 1. Session timer ──────────────────────────────────────────────────────────
import threading, time as _time

SESSION_START       = _time.time()
SESSION_LIMIT_HOURS = 11.5

def _timer_thread():
    while True:
        remaining = SESSION_LIMIT_HOURS - (_time.time() - SESSION_START) / 3600
        if remaining <= 1.5:
            print(f'\n WARNING: {remaining:.1f}h remaining — save checkpoints soon!')
        if remaining <= 0:
            print('\n Session limit reached.')
            break
        _time.sleep(1800)

threading.Thread(target=_timer_thread, daemon=True).start()
print(f'Session timer started. Limit: {SESSION_LIMIT_HOURS}h')

In [ ]:
# ── 2. Clone repos ────────────────────────────────────────────────────────────
import os, shutil, glob

# Reproduction repo (your fork with custom scripts)
if os.path.exists(REPO_DIR):
    print('Reproduction repo exists — pulling...')
    !cd {REPO_DIR} && git pull --quiet
else:
    !git clone {GITHUB_REPO} {REPO_DIR}

# Official HierarchicalDet repo (model code)
if os.path.exists(OFFICIAL_DIR):
    print('Official repo exists — skipping.')
else:
    !git clone https://github.com/ibrahimethemhamamci/HierarchicalDet {OFFICIAL_DIR}
    print('Official repo cloned.')

# Copy all reproduction scripts into official repo
patterns = ['*.py', '*.sh', '*.yaml', '*.md']
copied = []
for pat in patterns:
    for f in glob.glob(f'{REPO_DIR}/{pat}'):
        shutil.copy(f, OFFICIAL_DIR)
        copied.append(os.path.basename(f))
print(f'Copied {len(copied)} scripts to official repo.')

# Verify the critical scripts are present
critical = ['train_net_patched.py', 'dataset_mapper_patched.py', 'organize_dataset.py']
missing = [s for s in critical if not os.path.exists(f'{OFFICIAL_DIR}/{s}')]
if missing:
    raise RuntimeError(f'Critical scripts missing: {missing}\nCheck that GITHUB_REPO is correct and the repo is public.')
print('All critical scripts present.')

%cd {OFFICIAL_DIR}

In [ ]:
# ── 3. Install dependencies ───────────────────────────────────────────────────
# Kaggle pre-installs PyTorch and detectron2. We only need the extras.
import sys, time
t0 = time.time()

!pip install -q fvcore iopath omegaconf timm scipy termcolor yacs tabulate cloudpickle pycocotools huggingface_hub

# Add official repo to Python path
sys.path.insert(0, OFFICIAL_DIR)

import detectron2
print(f'detectron2: {detectron2.__version__}')
print(f'Install time: {time.time()-t0:.0f}s')

In [ ]:
# ── 4. Download Swin-B backbone ───────────────────────────────────────────────
import urllib.request, pickle, torch, time, os

os.makedirs(f'{OFFICIAL_DIR}/models', exist_ok=True)
pkl_path = f'{OFFICIAL_DIR}/models/swin_base_patch4_window7_224_22k.pkl'

if not os.path.exists(pkl_path):
    url = ('https://github.com/SwinTransformer/storage/releases/download/'
           'v1.0.0/swin_base_patch4_window7_224_22k.pth')
    pth_path = pkl_path.replace('.pkl', '.pth')
    print('Downloading Swin-B backbone (~340 MB)...')
    t0 = time.time()
    urllib.request.urlretrieve(url, pth_path)
    print(f'Downloaded in {time.time()-t0:.0f}s')
    ckpt = torch.load(pth_path, map_location='cpu')
    d2_ckpt = {'model': ckpt.get('model', ckpt),
               '__author__': 'third_party', 'matching_heuristics': True}
    with open(pkl_path, 'wb') as f:
        pickle.dump(d2_ckpt, f)
    print(f'Backbone saved: {pkl_path}')
else:
    print('Backbone already present.')

In [ ]:
# ── 5. Set up DENTEX dataset ──────────────────────────────────────────────────
import os, zipfile, time

DENTEX_RAW = f'{WORK_DIR}/sorted/dentex_raw/DENTEX'

def _symlink_sorted():
    """organize_dataset.py looks for sorted/ next to itself (in OFFICIAL_DIR).
    The actual data is in WORK_DIR/sorted, so symlink it in."""
    dst = f'{OFFICIAL_DIR}/sorted'
    src = f'{WORK_DIR}/sorted'
    if not os.path.exists(dst) and not os.path.islink(dst):
        os.symlink(src, dst)
        print(f'Linked {src} → {dst}')

if os.path.exists(f'{DATA_ROOT}/train_merged_disease_coco3class_onlyd_fixed.json'):
    print('Dataset already organized. Skipping.')

elif KAGGLE_DENTEX_INPUT:
    # Fast path: pre-uploaded Kaggle dataset
    print(f'Using mounted dataset: {KAGGLE_DENTEX_INPUT}')
    os.makedirs(DENTEX_RAW, exist_ok=True)
    import shutil
    val_json = f'{KAGGLE_DENTEX_INPUT}/validation_triple.json'
    if os.path.exists(val_json):
        shutil.copy(val_json, f'{DENTEX_RAW}/validation_triple.json')
    for zname in ['training_data.zip', 'validation_data.zip', 'test_data.zip']:
        zpath = f'{KAGGLE_DENTEX_INPUT}/{zname}'
        if os.path.exists(zpath):
            print(f'Extracting {zname}...')
            t0 = time.time()
            with zipfile.ZipFile(zpath, 'r') as zf:
                zf.extractall(f'{WORK_DIR}/sorted')
            print(f'  Done in {time.time()-t0:.0f}s')
        else:
            print(f'  WARNING: {zname} not found in {KAGGLE_DENTEX_INPUT}')
    _symlink_sorted()
    !python {OFFICIAL_DIR}/organize_dataset.py

else:
    # Slow path: download from HuggingFace
    from huggingface_hub import snapshot_download
    if HF_TOKEN:
        from huggingface_hub import login
        login(HF_TOKEN)
    print('Downloading DENTEX dataset (~11.8 GB)...')
    print('TIP: Upload zips as a Kaggle dataset to skip this step next time.')
    t0 = time.time()
    snapshot_download(
        repo_id='ibrahimhamamci/DENTEX',
        repo_type='dataset',
        local_dir=DENTEX_RAW,
        ignore_patterns=['*.git*', '.gitattributes'],
    )
    print(f'Download complete in {(time.time()-t0)/60:.1f} min')
    for zname in ['training_data.zip', 'validation_data.zip', 'test_data.zip']:
        zpath = f'{DENTEX_RAW}/{zname}'
        if os.path.exists(zpath):
            print(f'Extracting {zname}...')
            with zipfile.ZipFile(zpath, 'r') as zf:
                zf.extractall(f'{WORK_DIR}/sorted')
    _symlink_sorted()
    !python {OFFICIAL_DIR}/organize_dataset.py

print('\nDataset ready:')
!ls {DATA_ROOT}

In [ ]:
# ── 6. Restore checkpoints from previous session ──────────────────────────────
import shutil, os

os.makedirs(f'{OFFICIAL_DIR}/output/tier{TIER_TO_TRAIN}', exist_ok=True)

if PREV_OUTPUT_DIR:
    print(f'Restoring from: {PREV_OUTPUT_DIR}')
    for item in os.listdir(PREV_OUTPUT_DIR):
        src = os.path.join(PREV_OUTPUT_DIR, item)
        dst = os.path.join(f'{OFFICIAL_DIR}/output', item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
        print(f'  {item}')
    print('Restore complete.')
else:
    print('No previous output (fresh start).')

!find {OFFICIAL_DIR}/output -name '*.pth' 2>/dev/null | head -10 || echo 'No checkpoints yet.'

In [ ]:
# ── 7. Environment + config paths ─────────────────────────────────────────────
import os

os.environ['DATA_ROOT']       = DATA_ROOT
os.environ['DENTEX_TIER']     = f'tier{TIER_TO_TRAIN}'
os.environ['USE_NOISY_BOXES'] = '0'

CONFIG  = f'{OFFICIAL_DIR}/configs/diffdet.custom.swinbase.nonpretrain.yaml'

if TIER_TO_TRAIN == 1:
    WEIGHTS = f'{OFFICIAL_DIR}/models/swin_base_patch4_window7_224_22k.pkl'
elif TIER_TO_TRAIN == 2:
    WEIGHTS  = f'{OFFICIAL_DIR}/output/tier1/model_final.pth'
    NB_TRAIN = f'{OFFICIAL_DIR}/noisy_boxes/tier1_train_boxes.json'
    NB_VAL   = f'{OFFICIAL_DIR}/noisy_boxes/tier1_val_boxes.json'
    if os.path.exists(NB_TRAIN):
        os.environ['NOISY_BOX_TRAIN'] = NB_TRAIN
        os.environ['NOISY_BOX_VAL']   = NB_VAL
        os.environ['USE_NOISY_BOXES'] = '1'
elif TIER_TO_TRAIN == 3:
    WEIGHTS  = f'{OFFICIAL_DIR}/output/tier2/model_final.pth'
    NB_TRAIN = f'{OFFICIAL_DIR}/noisy_boxes/tier2_train_boxes.json'
    NB_VAL   = f'{OFFICIAL_DIR}/noisy_boxes/tier2_val_boxes.json'
    if os.path.exists(NB_TRAIN):
        os.environ['NOISY_BOX_TRAIN'] = NB_TRAIN
        os.environ['NOISY_BOX_VAL']   = NB_VAL
        os.environ['USE_NOISY_BOXES'] = '1'

assert os.path.exists(WEIGHTS), f'Weights not found: {WEIGHTS}'
assert os.path.exists(CONFIG),  f'Config not found: {CONFIG}'

print(f'Tier:            {TIER_TO_TRAIN}')
print(f'Weights:         {WEIGHTS}')
print(f'Config:          {CONFIG}')
print(f'USE_NOISY_BOXES: {os.environ["USE_NOISY_BOXES"]}')
print(f'DATA_ROOT:       {DATA_ROOT}')

In [ ]:
# ── 8. Generate noisy boxes (tiers 2 and 3 only) ─────────────────────────────
if TIER_TO_TRAIN > 1 and os.environ.get('USE_NOISY_BOXES') != '1':
    prev = TIER_TO_TRAIN - 1
    prev_weights = f'{OFFICIAL_DIR}/output/tier{prev}/model_final.pth'
    assert os.path.exists(prev_weights), f'Tier {prev} checkpoint not found: {prev_weights}'
    os.makedirs(f'{OFFICIAL_DIR}/noisy_boxes', exist_ok=True)
    for split, ann_json, img_dir, out_file in [
        ('train',
         f'{DATA_ROOT}/train_merged_disease_coco3class_onlyd_fixed.json',
         f'{DATA_ROOT}/for_coco_disease_train',
         f'{OFFICIAL_DIR}/noisy_boxes/tier{prev}_train_boxes.json'),
        ('val',
         f'{DATA_ROOT}/test_merged_disease_coco3class.json',
         f'{DATA_ROOT}/for_coco_disease_test',
         f'{OFFICIAL_DIR}/noisy_boxes/tier{prev}_val_boxes.json'),
    ]:
        print(f'Generating noisy boxes ({split})...')
        !python {OFFICIAL_DIR}/phase2_generate_noisy_boxes.py \
            --config-file {CONFIG} \
            --weights {prev_weights} \
            --tier {prev - 1} \
            --ann-json {ann_json} \
            --img-dir {img_dir} \
            --out {out_file}
    os.environ['NOISY_BOX_TRAIN'] = f'{OFFICIAL_DIR}/noisy_boxes/tier{prev}_train_boxes.json'
    os.environ['NOISY_BOX_VAL']   = f'{OFFICIAL_DIR}/noisy_boxes/tier{prev}_val_boxes.json'
    os.environ['USE_NOISY_BOXES'] = '1'
    print('Noisy boxes generated.')
else:
    print(f'Tier {TIER_TO_TRAIN}: noisy box step skipped.')

In [ ]:
# ── 9. Train ──────────────────────────────────────────────────────────────────
out_dir    = f'{OFFICIAL_DIR}/output/tier{TIER_TO_TRAIN}'
final_ckpt = f'{out_dir}/model_final.pth'
os.makedirs(out_dir, exist_ok=True)

if os.path.exists(final_ckpt):
    print(f'model_final.pth already exists — tier {TIER_TO_TRAIN} is done.')
    print('Proceed to the evaluate cell, or set TIER_TO_TRAIN to the next tier.')
else:
    print(f'Starting tier {TIER_TO_TRAIN} training ({MAX_ITER} iters)...')
    print(f'Estimated: 8–12h on T4.  Session limit: {SESSION_LIMIT_HOURS}h')
    print('─' * 60)
    !python {OFFICIAL_DIR}/train_net_patched.py \
        --config-file {CONFIG} \
        --num-gpus {NUM_GPUS} \
        MODEL.WEIGHTS {WEIGHTS} \
        OUTPUT_DIR {out_dir} \
        SOLVER.MAX_ITER {MAX_ITER} \
        SEED 40244023
    if os.path.exists(final_ckpt):
        size_mb = os.path.getsize(final_ckpt) / 1e6
        print(f'\n Training complete! {final_ckpt} ({size_mb:.0f} MB)')
    else:
        print('\n Training stopped before completion (session limit or error).')
        print('Save the output directory and resume in a new session.')

In [ ]:
# ── 10. Evaluate ──────────────────────────────────────────────────────────────
final_ckpt = f'{OFFICIAL_DIR}/output/tier{TIER_TO_TRAIN}/model_final.pth'
if not os.path.exists(final_ckpt):
    print('No final checkpoint — run training cell first.')
else:
    eval_out = f'{OFFICIAL_DIR}/output/tier{TIER_TO_TRAIN}_eval'
    os.makedirs(eval_out, exist_ok=True)
    print(f'Evaluating tier {TIER_TO_TRAIN}...')
    !python {OFFICIAL_DIR}/train_net_patched.py \
        --config-file {CONFIG} \
        --eval-only \
        MODEL.WEIGHTS {final_ckpt} \
        OUTPUT_DIR {eval_out}
    !python {OFFICIAL_DIR}/phase2_collect_results.py \
        --log-files {eval_out}/log.txt \
        --tier-names "Tier{TIER_TO_TRAIN}"

In [ ]:
# ── 11. Save outputs for next session ─────────────────────────────────────────
# Copies checkpoints into /kaggle/working/ which Kaggle saves as notebook output.
# After this cell: click Save Version → Save & Run All (select 'Only files').
import os, shutil

for src, dst in [
    (f'{OFFICIAL_DIR}/output',      f'{WORK_DIR}/output'),
    (f'{OFFICIAL_DIR}/noisy_boxes', f'{WORK_DIR}/noisy_boxes'),
]:
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Saved: {dst}')

print('\n=== Saved files ===')
for root, _, files in os.walk(f'{WORK_DIR}/output'):
    for f in files:
        path = os.path.join(root, f)
        print(f'  {path.replace(WORK_DIR,"")}  ({os.path.getsize(path)/1e6:.0f} MB)')

next_tier = TIER_TO_TRAIN + 1
if next_tier <= 3:
    print(f'\n=== Next session ===')
    print(f'1. Save Version → Save & Run All → Only files')
    print(f'2. New session → Add Data → this notebook output')
    print(f'3. Set TIER_TO_TRAIN = {next_tier}')
    print(f'4. Set PREV_OUTPUT_DIR = "/kaggle/input/<dataset-name>/output"')
    print(f'5. Run All')
else:
    print('\nAll 3 tiers complete! Proceed to evaluation and extensions.')

In [ ]:
# ── 12. (Optional) Runtime benchmark ─────────────────────────────────────────
final_ckpt = f'{OFFICIAL_DIR}/output/tier{TIER_TO_TRAIN}/model_final.pth'
if not os.path.exists(final_ckpt):
    print('No checkpoint — skipping benchmark.')
else:
    for device, n in [('cuda', 20), ('cpu', 5)]:
        out = f'{WORK_DIR}/output/runtime_tier{TIER_TO_TRAIN}_{device}.json'
        !python {OFFICIAL_DIR}/phase2_runtime_benchmark.py \
            --config-file {CONFIG} \
            --weights {final_ckpt} \
            --tier {TIER_TO_TRAIN - 1} \
            --n-images {n} \
            --device {device} \
            --out {out}